# Pause: resumable Colab feature extraction and training
No dataset or model accuracy is claimed by this notebook. Obtain official dataset access and verify licence rights first. Process small batches; save checkpoints to Drive. A T4 is optional: this baseline uses CPU MediaPipe and scikit-learn. Do not upload private app sessions or credentials.

For all 56 UBFC-Phys participants, training uses 28 and testing uses 28. If you acquire only 28, training uses 14 and testing uses 14. Do not choose participants after looking at test accuracy.


In [ ]:
%pip install mediapipe opencv-python-headless numpy scikit-learn


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, urllib.request, subprocess, sys
# Authorize your own Drive in Colab; no credential belongs in notebook source.
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/pause-training')
ROOT.mkdir(parents=True,exist_ok=True)
LOCAL=Path('/content/pause-work');LOCAL.mkdir(exist_ok=True)
print('Temporary free GiB:',round(shutil.disk_usage('/content').free/2**30,1))


## Authorized data and reviewed labels
Use official download instructions at https://sites.google.com/view/ybenezeth/ubfc-phys . No login bypass or guessed links are included. Download one participant/small batch to `/content/pause-work` through authorized access. Do not try storing the entire video collection in free Colab or Drive.

Create a CSV manifest with columns `participant_id,video_path,label`. Paths point to currently available videos. Label 0/1 must follow a reviewed protocol. Rest versus stressor *task conditions* are only proxies. Check experiment group/difficulty metadata; do not automatically label every speech/arithmetic video stressed. No per-frame label can be inferred from a pre/post anxiety questionnaire.

Use the same immutable video paths and manifest contents when resuming. Checkpoints key the manifest row and landmark model hash; replace/checkpoint directory if source video contents change. Never count renamed copies twice.


In [ ]:
MODEL=LOCAL/'face_landmarker.task'
if not MODEL.exists():
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task',MODEL)
MANIFEST=ROOT/'manifest.csv'
LICENSE_NOTE='' # Describe verified research rights / permission.
LABEL_DEFINITION='' # Describe the actual reviewed label protocol.
assert MANIFEST.exists(), 'Create the reviewed manifest.csv in pause-training first.'
assert LICENSE_NOTE and LABEL_DEFINITION, 'Document licence and labels before continuing.'


## Extraction code
The next cell writes the included extractor. It matches the current app’s three means, 30-second windows, approximate 10 Hz sampling, and provisional quality thresholds. It rejects incomplete/poor windows. Python/browser parity must still be verified. No pulse or wearable inputs are used.

In [ ]:
EXTRACT_CODE = '"""Resumable local-video feature extraction; labels come from a reviewed manifest."""\nimport argparse, csv, hashlib, json, math\nfrom pathlib import Path\nimport cv2\nimport mediapipe as mp\nimport numpy as np\n\ndef quality(points, count, brightness, previous):\n    if count != 1 or not points:\n        return False, None, 0\n    xs=[p.x for p in points]; ys=[p.y for p in points]\n    w=max(xs)-min(xs); h=max(ys)-min(ys)\n    center=((max(xs)+min(xs))/2,(max(ys)+min(ys))/2)\n    movement=math.dist(center, previous) if previous else 0\n    good=(45 <= brightness <= 225 and .18 <= w <= .8 and .22 <= h <= .9\n          and abs(center[0]-.5)<=.2 and abs(center[1]-.5)<=.22 and movement<=.035)\n    return good, center, movement\n\ndef extract(path, model_path):\n    cap=cv2.VideoCapture(str(path))\n    fps=cap.get(cv2.CAP_PROP_FPS)\n    if not cap.isOpened() or not math.isfinite(fps) or fps<=0:\n        cap.release(); raise ValueError(\'Unreadable video or invalid FPS\')\n    options=mp.tasks.vision.FaceLandmarkerOptions(\n        base_options=mp.tasks.BaseOptions(model_asset_path=str(model_path)),\n        running_mode=mp.tasks.vision.RunningMode.VIDEO,num_faces=2,output_face_blendshapes=True)\n    windows={}; previous=None; index=0; next_sample=0\n    try:\n        with mp.tasks.vision.FaceLandmarker.create_from_options(options) as model:\n            while True:\n                success, frame=cap.read()\n                if not success: break\n                seconds=index/fps; index+=1\n                if seconds+1e-8<next_sample: continue\n                next_sample=seconds+.1\n                rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)\n                result=model.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb),int(seconds*1000))\n                small=cv2.resize(rgb,(64,48),interpolation=cv2.INTER_LINEAR)\n                points=result.face_landmarks[0] if result.face_landmarks else None\n                good,previous,movement=quality(points,len(result.face_landmarks),float(small.mean()),previous)\n                window=windows.setdefault(int(seconds//30),{\'total\':0,\'samples\':[]})\n                window[\'total\']+=1\n                if good:\n                    shapes={c.category_name:c.score for c in result.face_blendshapes[0]}\n                    window[\'samples\'].append([(shapes.get(\'eyeBlinkLeft\',0)+shapes.get(\'eyeBlinkRight\',0))/2,\n                        (shapes.get(\'browDownLeft\',0)+shapes.get(\'browDownRight\',0))/2,movement])\n    finally: cap.release()\n    duration=index/fps\n    output=[]\n    for number,w in windows.items():\n        if duration < (number+1)*30: continue # discard incomplete windows\n        ratio=len(w[\'samples\'])/max(w[\'total\'],1)\n        good=ratio>=.75 and len(w[\'samples\'])>=30\n        output.append({\'window\':number,\'usable_fraction\':ratio,\'accepted\':good,\n                       \'features\':np.mean(w[\'samples\'],axis=0).tolist() if good else None})\n    return output\n\ndef main():\n    p=argparse.ArgumentParser(); p.add_argument(\'--manifest\',type=Path,required=True)\n    p.add_argument(\'--model\',type=Path,required=True);p.add_argument(\'--checkpoint\',type=Path,required=True)\n    p.add_argument(\'--csv\',type=Path,required=True);a=p.parse_args()\n    a.checkpoint.mkdir(parents=True,exist_ok=True)\n    with a.manifest.open() as handle: rows=list(csv.DictReader(handle))\n    if not rows or not {\'participant_id\',\'video_path\',\'label\'}.issubset(rows[0]):\n        raise ValueError(\'Manifest requires participant_id,video_path,label\')\n    if any(r[\'label\'] not in [\'0\',\'1\'] or not r[\'participant_id\'] for r in rows):\n        raise ValueError(\'Review all participant IDs and binary labels first\')\n    seen=set(); all_rows=[]; rejected=0\n    for row in rows:\n        identity=json.dumps(row,sort_keys=True)\n        if identity in seen: raise ValueError(\'Duplicate manifest row\')\n        seen.add(identity)\n        key=hashlib.sha256((identity+hashlib.sha256(a.model.read_bytes()).hexdigest()).encode()).hexdigest()\n        checkpoint=a.checkpoint/(key+\'.json\')\n        if checkpoint.exists(): windows=json.loads(checkpoint.read_text())\n        else:\n            windows=extract(Path(row[\'video_path\']),a.model)\n            temp=checkpoint.with_suffix(\'.tmp\');temp.write_text(json.dumps(windows));temp.replace(checkpoint)\n        for w in windows:\n            if not w[\'accepted\']: rejected+=1;continue\n            all_rows.append([row[\'participant_id\'],row[\'label\'],*w[\'features\']])\n        a.csv.parent.mkdir(parents=True,exist_ok=True)\n        with a.csv.open(\'w\',newline=\'\') as handle:\n            writer=csv.writer(handle);writer.writerow([\'participant_id\',\'label\',\'meanBlinkCoefficient\',\'meanBrowCoefficient\',\'meanMovement\']);writer.writerows(all_rows)\n        print(\'Completed manifest videos:\',len(seen),\'Accepted windows:\',len(all_rows),\'Rejected:\',rejected)\n    (a.csv.parent/\'extraction_summary.json\').write_text(json.dumps({\'accepted_windows\':len(all_rows),\'rejected_windows\':rejected,\'sampling_hz_target\':10,\'window_seconds\':30,\'model_sha256\':hashlib.sha256(a.model.read_bytes()).hexdigest(),\'limitations\':\'Python/JS MediaPipe implementations and image resampling can differ; validate parity before deployment.\'},indent=2))\nif __name__==\'__main__\': main()\n'
(LOCAL/'extract.py').write_text(EXTRACT_CODE)


In [ ]:
subprocess.run([sys.executable,str(LOCAL/'extract.py'),'--manifest',str(MANIFEST),'--model',str(MODEL),'--checkpoint',str(ROOT/'checkpoints'),'--csv',str(ROOT/'features.csv')],check=True)
# Keep checkpoints and CSV on Drive. Remove only temporary video files you own
# after processing. Fetch the next batch and resume with the full manifest when
# earlier video paths no longer exist: their checkpoints supply their features.


## Train and evaluate
Run after extraction includes at least eight people with both class conditions. Hyperparameters are selected via grouped CV inside the training half. Test results are evaluated once. Reports include a dummy baseline and participant-bootstrap interval. Brier score is probability error, not proof of calibrated probabilities. Repeated test-set tuning invalidates the benchmark.

In [ ]:
TRAIN_CODE = '"""Train on half the participants; evaluate on the untouched other half.\n\nInput must be real extracted video features, not synthetic observations.\nNo pickle loading; model exports numeric coefficients and documented metrics.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, brier_score_loss\nfrom sklearn.model_selection import GroupKFold, cross_val_score\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nFEATURES = [\'meanBlinkCoefficient\', \'meanBrowCoefficient\', \'meanMovement\']\n\ndef metrics(y, prediction, probability):\n    return {\'accuracy\': accuracy_score(y, prediction),\n            \'balanced_accuracy\': balanced_accuracy_score(y, prediction),\n            \'precision\': precision_score(y, prediction, zero_division=0),\n            \'recall\': recall_score(y, prediction, zero_division=0),\n            \'f1\': f1_score(y, prediction, zero_division=0),\n            \'confusion_matrix_labels\': [0, 1],\n            \'confusion_matrix\': confusion_matrix(y, prediction, labels=[0, 1]).tolist(),\n            \'roc_auc\': roc_auc_score(y, probability),\n            \'brier_score\': brier_score_loss(y, probability)}\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--csv\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, default=Path(\'artifacts\'))\n    parser.add_argument(\'--dataset-name\', required=True)\n    parser.add_argument(\'--label-definition\', required=True)\n    parser.add_argument(\'--license-note\', required=True)\n    parser.add_argument(\'--seed\', type=int, default=42)\n    args = parser.parse_args()\n    with args.csv.open() as handle:\n        rows = list(csv.DictReader(handle))\n    required = {\'participant_id\', \'label\', *FEATURES}\n    if not rows or not required.issubset(rows[0]):\n        raise ValueError(\'Required columns: \' + \', \'.join(sorted(required)))\n    groups = np.array([row[\'participant_id\'] for row in rows])\n    y = np.array([int(row[\'label\']) for row in rows])\n    x = np.array([[float(row[f]) for f in FEATURES] for row in rows])\n    if not np.isfinite(x).all() or set(y) != {0, 1}:\n        raise ValueError(\'Require finite features and both binary labels 0 and 1\')\n    if not ((x[:, :2] >= 0) & (x[:, :2] <= 1)).all() or (x[:, 2] < 0).any():\n        raise ValueError(\'Features must match the browser coefficient definitions\')\n    participants = np.unique(groups)\n    if len(participants) < 8:\n        raise ValueError(\'Need at least 8 participants for this experimental split\')\n    rng = np.random.default_rng(args.seed)\n    rng.shuffle(participants)\n    train_people = participants[:len(participants)//2]\n    train = np.isin(groups, train_people)\n    test = ~train\n    if set(y[train]) != {0, 1} or set(y[test]) != {0, 1}:\n        raise ValueError(\'Both participant partitions must contain both labels\')\n    cv = GroupKFold(n_splits=min(5, len(train_people)))\n    folds = list(cv.split(x[train], y[train], groups[train]))\n    if any(set(y[train][a]) != {0,1} or set(y[train][b]) != {0,1} for a,b in folds):\n        raise ValueError(\'CV folds must contain both labels; review participant coverage\')\n    candidates = []\n    for c in [.01, .1, 1, 10]:\n        model = make_pipeline(StandardScaler(), LogisticRegression(C=c, class_weight=\'balanced\', max_iter=2000, random_state=args.seed))\n        score = float(cross_val_score(model, x[train], y[train], cv=folds, scoring=\'balanced_accuracy\').mean())\n        candidates.append((score, c, model))\n    cv_score, c, model = max(candidates, key=lambda item: item[0])\n    model.fit(x[train], y[train])\n    probabilities = model.predict_proba(x[test])[:, 1]\n    result = metrics(y[test], model.predict(x[test]), probabilities)\n    dummy = DummyClassifier(strategy=\'prior\').fit(x[train], y[train])\n    baseline = metrics(y[test], dummy.predict(x[test]), dummy.predict_proba(x[test])[:, 1])\n    # Bootstrap whole held-out people, preserving within-person dependence.\n    held_out = participants[len(train_people):]\n    boot = []\n    prediction = model.predict(x[test]); test_y=y[test]; test_groups=groups[test]\n    for _ in range(1000):\n        sampled = rng.choice(held_out, len(held_out), replace=True)\n        index = np.concatenate([np.flatnonzero(test_groups == person) for person in sampled])\n        if len(np.unique(test_y[index])) == 2:\n            boot.append(balanced_accuracy_score(test_y[index], prediction[index]))\n    scaler, logistic = model.steps[0][1], model.steps[1][1]\n    report = {\'dataset\': args.dataset_name, \'label_definition\': args.label_definition,\n              \'license_note\': args.license_note, \'input_sha256\': hashlib.sha256(args.csv.read_bytes()).hexdigest(),\n              \'seed\': args.seed, \'features\': FEATURES, \'train_participants\': train_people.tolist(),\n              \'test_participants\': held_out.tolist(), \'train_rows\': int(train.sum()), \'test_rows\': int(test.sum()),\n              \'train_fraction_participants\': len(train_people)/len(participants),\n              \'selected_C\': c, \'training_group_cv_balanced_accuracy\': cv_score,\n              \'held_out_metrics\': result, \'dummy_baseline\': baseline,\n              \'balanced_accuracy_95pct_participant_bootstrap\': np.quantile(boot,[.025,.975]).tolist() if boot else None,\n              \'limitations\': [\'Laboratory task labels may be proxies for stress.\', \'Facial behavior can reveal speech/task differences rather than stress.\', \'No real-world or clinical validity established.\', \'Do not tune this model against the held-out test set.\']}\n    artifact = {\'features\': FEATURES, \'mean\': scaler.mean_.tolist(), \'scale\': scaler.scale_.tolist(),\n                \'coefficients\': logistic.coef_[0].tolist(), \'intercept\': float(logistic.intercept_[0]),\n                \'threshold\': .5, \'experimental\': True, \'report\': \'metrics.json\'}\n    args.output.mkdir(parents=True, exist_ok=True)\n    (args.output/\'metrics.json\').write_text(json.dumps(report, indent=2))\n    (args.output/\'model.json\').write_text(json.dumps(artifact, indent=2))\n    print(json.dumps({\'held_out_metrics\': result, \'dummy_baseline\': baseline}, indent=2))\n\nif __name__ == \'__main__\':\n    main()\n'
(LOCAL/'train.py').write_text(TRAIN_CODE)


In [ ]:
subprocess.run([sys.executable,str(LOCAL/'train.py'),'--csv',str(ROOT/'features.csv'),'--dataset-name','UBFC-Phys','--label-definition',LABEL_DEFINITION,'--license-note',LICENSE_NOTE,'--output',str(ROOT/'artifacts')],check=True)
import json
report=json.loads((ROOT/'artifacts/metrics.json').read_text())
print(json.dumps(report['held_out_metrics'],indent=2))
print('Baseline:',report['dummy_baseline'])
print('Confidence interval:',report['balanced_accuracy_95pct_participant_bootstrap'])


## Download compact outputs
Download the small model/report, not raw videos. App inference integration remains pending until actual metrics and extraction parity are reviewed. Laboratory task prediction is not clinical or real-world stress accuracy.

In [ ]:
from google.colab import files
archive=shutil.make_archive('/content/pause-model','zip',ROOT/'artifacts')
files.download(archive)
# Optional: files.download(str(ROOT/'features.csv'))
